[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/093.%20The%20Inventory-Routing%20Problem%20%28IRP%29/P93-Tier-2_executed.ipynb)

---

# 93. The Inventory-Routing Problem (IRP)
## Tier 2: A GRASP Heuristic

### Purpose and scope

The Inventory-Routing Problem (IRP) combines two decisions: how much product to deliver to each customer and which vehicle routes to use. This notebook illustrates a Greedy Randomized Adaptive Search Procedure (GRASP) for a small, multi-period instance. It constructs routes using a Restricted Candidate List (RCL), improves each constructed route with 2-opt local search, and retains the best solution found across repeated iterations.

The example uses one depot, eight customers, two vehicles, and a five-period planning horizon. Customer demand is sampled from a normal distribution and clipped at zero. The model uses Euclidean distances, fixed and distance-based vehicle costs, customer and depot holding costs, and a fixed penalty for ending a period below a customer's minimum inventory.

This is an illustrative heuristic, not an exact optimizer or a complete operational IRP model. Deliveries are limited to 10 units per customer per period; vehicle capacity is measured in units delivered; each customer can appear at most once in a period's constructed routes; and depot inventory limits total deliveries. The model does not include delivery time windows, vehicle travel-time limits, backorders, or an explicit service-level constraint. Demand realizations differ between GRASP iterations, so the lowest recorded cost is the best observed scenario-dependent cost, not an estimate of expected cost. GRASP does not guarantee a globally optimal solution.

### How the heuristic works

1. **Construction:** For each vehicle, score feasible unvisited customers using delivery urgency and the cost of inserting the customer into the current route. The RCL contains a fraction of the best-scoring candidates, controlled by `alpha`; a candidate is selected randomly from that list.
2. **Local search:** Apply 2-opt moves, which reverse a segment of a route, accepting a move when it reduces that route's transportation cost. Repeat until no tested move improves the route.
3. **Iteration:** Construct and improve a solution for every planning period, simulate demand and update inventories, then calculate the total cost. Retain the least-cost solution encountered.

Here, a smaller `alpha` makes candidate selection more restricted, while a larger value admits more candidates and adds more random exploration. The implementation uses 2-opt; it does not implement Or-opt or swap operators.

### Mathematical description

Let $t=1,\ldots,T$ index periods, $i$ index customers, and $k$ index vehicles. Let $I_{i,t}$ be customer $i$'s inventory at the start of period $t$, $q_{i,t}$ its delivery, and $D_{i,t}$ its nonnegative demand realization. Let $I_{0,t}$ denote depot inventory. Deliveries occur before demand is applied, so the end-of-period inventories are

$$I_{i,t+1}=\max\left(0,\ I_{i,t}+q_{i,t}-D_{i,t}\right),\qquad I_{0,t+1}=I_{0,t}-\sum_i q_{i,t}.$$

The zero lower bound means unmet demand is not carried as negative inventory; it is not a separate lost-sales decision. The constructed deliveries satisfy $0\le q_{i,t}\le 10$, $I_{i,t}+q_{i,t}\le I_i^{\max}$, and the sum delivered on a vehicle's route does not exceed that vehicle's capacity. Total deliveries in a period cannot exceed the available depot inventory. A customer is included at most once in that period's routes.

For a nonempty route $r$ assigned to vehicle $k$, the transportation cost is

$$C_{r}=F_k+c_k d_r,$$

where $F_k$ is the vehicle's fixed route cost, $c_k$ is its cost per distance unit, and $d_r$ is the Euclidean distance from the depot through the route's customer sequence and back to the depot. The total cost evaluated by the heuristic is

$$C=\sum_{t=1}^{T}\sum_{r\in\mathcal{R}_t} C_r
+\sum_{t=1}^{T}\left(h_0 I_{0,t+1}+\sum_i h_i I_{i,t+1}
+P\sum_i \mathbf{1}\{I_{i,t+1}<I_i^{\min}\}\right).$$

Here, $\mathcal{R}_t$ is the set of constructed routes in period $t$, $h_0$ and $h_i$ are depot and customer holding-cost rates, $I_i^{\min}$ is the customer's minimum inventory, and $P=1000$ is the stockout-threshold penalty used in the code. The indicator $\mathbf{1}\{\cdot\}$ equals one when its condition is true and zero otherwise. The initial inventory is not charged as an end-of-period holding cost; the cost is assessed on inventory after each period's delivery and demand.

The construction score is the insertion-route cost divided by one plus the inventory urgency score. Urgency increases as inventory approaches the minimum stock level. The RCL is formed from the best-scoring candidates, and random selection from that list supplies GRASP's construction diversity. The implementation is a heuristic rather than a solver for the cost expression above: its construction and 2-opt choices are local decisions, and it does not search all feasible policies.

If a period has $n$ customers and $m$ vehicles, candidate scoring considers unvisited customers and tests insertion positions in the current route. A single insertion test is proportional to route length because route cost is recalculated. The 2-opt search tests pairs of route positions and can require repeated passes until no improvement is found. Consequently, runtime depends on the number of customers, vehicles, periods, and GRASP iterations; no optimality or fixed runtime guarantee follows from the heuristic.

In [ ]:
from dataclasses import dataclass
import random
import time
from typing import Dict, List, Optional, Set, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class Customer:
    """Data describing a customer and its inventory characteristics."""

    id: int
    x: float
    y: float
    demand_mean: float
    demand_std: float
    holding_cost: float
    initial_inventory: float
    min_inventory: float
    max_inventory: float


@dataclass
class Depot:
    """Location, starting inventory, and holding cost of the depot."""

    x: float
    y: float
    initial_inventory: float
    holding_cost: float


@dataclass
class Vehicle:
    """Capacity and transportation-cost data for a vehicle."""

    id: int
    capacity: float
    fixed_cost: float
    variable_cost: float


@dataclass
class Delivery:
    """Quantity delivered to one customer in one planning period."""

    customer_id: int
    quantity: float
    period: int


@dataclass
class Route:
    """A vehicle's ordered customer visits and associated deliveries."""

    vehicle_id: int
    customers: List[int]
    deliveries: List[Delivery]
    total_distance: float
    total_cost: float

In [ ]:
class GRASPIRP:
    """GRASP construction and 2-opt local search for a simplified IRP."""

    def __init__(
        self,
        depot: Depot,
        customers: List[Customer],
        vehicles: List[Vehicle],
        num_periods: int,
        alpha: float = 0.3,
        max_iterations: int = 100,
    ):
        if not 0.0 < alpha <= 1.0:
            raise ValueError("alpha must be greater than 0 and at most 1.")
        if num_periods <= 0 or max_iterations <= 0:
            raise ValueError("num_periods and max_iterations must be positive.")
        if any(customer.id == 0 for customer in customers):
            raise ValueError("Customer IDs must not use 0; it identifies the depot.")
        if len({customer.id for customer in customers}) != len(customers):
            raise ValueError("Customer IDs must be unique.")

        self.depot = depot
        self.customers = customers
        self.vehicles = vehicles
        self.num_periods = num_periods
        self.alpha = alpha
        self.max_iterations = max_iterations
        self.distances = self._compute_distances()

    def _compute_distances(self) -> Dict[Tuple[int, int], float]:
        """Precompute Euclidean distances for route evaluation."""
        distances = {}

        for customer in self.customers:
            distance = np.hypot(self.depot.x - customer.x, self.depot.y - customer.y)
            distances[(0, customer.id)] = distance
            distances[(customer.id, 0)] = distance

        for customer_a in self.customers:
            for customer_b in self.customers:
                if customer_a.id != customer_b.id:
                    distances[(customer_a.id, customer_b.id)] = np.hypot(
                        customer_a.x - customer_b.x,
                        customer_a.y - customer_b.y,
                    )

        return distances

    def _route_distance(self, customers: List[int]) -> float:
        """Measure a closed route from the depot through its customers."""
        if not customers:
            return 0.0

        total_distance = self.distances[(0, customers[0])]
        for index in range(len(customers) - 1):
            total_distance += self.distances[(customers[index], customers[index + 1])]
        total_distance += self.distances[(customers[-1], 0)]
        return total_distance

    def _route_cost(self, route: List[int], vehicle: Vehicle) -> float:
        """Apply fixed and distance-based costs to a nonempty route."""
        if not route:
            return 0.0
        return vehicle.fixed_cost + vehicle.variable_cost * self._route_distance(route)

    def _inventory_urgency(
        self,
        customer: Customer,
        current_inventory: float,
        period: int,
    ) -> float:
        """Give higher scores to customers closer to their minimum stock."""
        del period  # The score uses current stock, not a period-specific forecast.
        if current_inventory <= customer.min_inventory:
            return 1000.0

        days_of_stock = (current_inventory - customer.min_inventory) / max(
            customer.demand_mean, 1e-12
        )
        return 10.0 / (days_of_stock + 1.0)

    def _distance_cost(
        self,
        customer: Customer,
        current_route: List[int],
        vehicle: Vehicle,
    ) -> float:
        """Find the least-cost route obtained by inserting a customer."""
        best_cost = float("inf")
        for position in range(len(current_route) + 1):
            test_route = current_route.copy()
            test_route.insert(position, customer.id)
            best_cost = min(best_cost, self._route_cost(test_route, vehicle))
        return best_cost

    def _create_rcl(
        self,
        unvisited: List[int],
        current_route: List[int],
        vehicle: Vehicle,
        inventories: Dict[int, float],
        period: int,
        depot_available: float,
        route_load: float,
    ) -> List[int]:
        """Rank feasible customers and return the restricted candidate list."""
        candidates = []
        customers_by_id = {customer.id: customer for customer in self.customers}

        for customer_id in unvisited:
            customer = customers_by_id[customer_id]
            delivery_quantity = min(
                10.0,
                customer.max_inventory - inventories[customer_id],
                vehicle.capacity - route_load,
                depot_available,
            )
            if delivery_quantity <= 0.0:
                continue

            urgency = self._inventory_urgency(
                customer, inventories[customer_id], period
            )
            insertion_cost = self._distance_cost(customer, current_route, vehicle)
            score = insertion_cost / (urgency + 1.0)
            candidates.append((customer_id, score))

        if not candidates:
            return []

        candidates.sort(key=lambda candidate: candidate[1])
        rcl_size = max(1, int(len(candidates) * self.alpha))
        return [customer_id for customer_id, _ in candidates[:rcl_size]]

    def _construct_solution(
        self,
        period: int,
        inventories: Dict[int, float],
    ) -> List[Route]:
        """Build period routes by repeatedly selecting from an RCL."""
        routes = []
        unvisited = [customer.id for customer in self.customers]
        depot_available = max(0.0, inventories[0])

        for vehicle in self.vehicles:
            if not unvisited or depot_available <= 0.0:
                break

            current_route = []
            deliveries = []
            route_load = 0.0

            while unvisited and depot_available > 0.0:
                rcl = self._create_rcl(
                    unvisited=unvisited,
                    current_route=current_route,
                    vehicle=vehicle,
                    inventories=inventories,
                    period=period,
                    depot_available=depot_available,
                    route_load=route_load,
                )
                if not rcl:
                    break

                selected_customer = random.choice(rcl)
                customer = next(
                    item for item in self.customers if item.id == selected_customer
                )
                delivery_quantity = min(
                    10.0,
                    customer.max_inventory - inventories[selected_customer],
                    vehicle.capacity - route_load,
                    depot_available,
                )
                if delivery_quantity <= 0.0:
                    unvisited.remove(selected_customer)
                    continue

                current_route.append(selected_customer)
                unvisited.remove(selected_customer)
                deliveries.append(
                    Delivery(selected_customer, delivery_quantity, period)
                )
                route_load += delivery_quantity
                depot_available -= delivery_quantity

            if current_route:
                total_distance = self._route_distance(current_route)
                routes.append(
                    Route(
                        vehicle_id=vehicle.id,
                        customers=current_route,
                        deliveries=deliveries,
                        total_distance=total_distance,
                        total_cost=self._route_cost(current_route, vehicle),
                    )
                )

        return routes

    def _local_search_2opt(self, routes: List[Route]) -> List[Route]:
        """Repeatedly accept route reversals that reduce transportation cost."""
        vehicles_by_id = {vehicle.id: vehicle for vehicle in self.vehicles}
        improved = True

        while improved:
            improved = False
            for route in routes:
                if len(route.customers) < 2:
                    continue

                vehicle = vehicles_by_id[route.vehicle_id]
                for first in range(len(route.customers) - 1):
                    for last in range(first + 1, len(route.customers)):
                        new_customers = route.customers.copy()
                        new_customers[first:last + 1] = reversed(
                            new_customers[first:last + 1]
                        )
                        new_cost = self._route_cost(new_customers, vehicle)
                        if new_cost < route.total_cost - 1e-12:
                            route.customers = new_customers
                            route.total_distance = self._route_distance(new_customers)
                            route.total_cost = new_cost
                            improved = True
                            break
                    if improved:
                        break
                if improved:
                    break

        return routes

    def _calculate_total_cost(
        self,
        solution: List[List[Route]],
        inventories_history: List[Dict[int, float]],
    ) -> float:
        """Sum route, end-of-period holding, and minimum-stock penalty costs."""
        total_cost = sum(
            route.total_cost
            for period_routes in solution
            for route in period_routes
        )

        for inventories in inventories_history[1:]:
            total_cost += inventories[0] * self.depot.holding_cost
            for customer in self.customers:
                inventory = inventories[customer.id]
                total_cost += inventory * customer.holding_cost
                if inventory < customer.min_inventory:
                    total_cost += 1000.0

        return total_cost

    def solve(self) -> Tuple[List[List[Route]], float, List[float]]:
        """Run GRASP iterations and return the best solution and cost history."""
        best_solution = None
        best_cost = float("inf")
        cost_history = []

        print(f"Starting GRASP with {self.max_iterations} iterations...")
        for iteration in range(self.max_iterations):
            inventories = {0: self.depot.initial_inventory}
            inventories.update(
                {
                    customer.id: customer.initial_inventory
                    for customer in self.customers
                }
            )
            inventories_history = [inventories.copy()]
            solution = []

            for period in range(self.num_periods):
                period_routes = self._construct_solution(period, inventories)
                period_routes = self._local_search_2opt(period_routes)
                solution.append(period_routes)

                for route in period_routes:
                    for delivery in route.deliveries:
                        inventories[delivery.customer_id] += delivery.quantity
                        inventories[0] -= delivery.quantity

                for customer in self.customers:
                    demand = max(
                        0.0,
                        np.random.normal(customer.demand_mean, customer.demand_std),
                    )
                    inventories[customer.id] = max(
                        0.0, inventories[customer.id] - demand
                    )

                inventories_history.append(inventories.copy())

            total_cost = self._calculate_total_cost(solution, inventories_history)
            if total_cost < best_cost:
                best_cost = total_cost
                best_solution = solution
                print(f"Iteration {iteration + 1}: New best cost = ${best_cost:.2f}")
            cost_history.append(best_cost)

        print(f"GRASP completed. Best observed cost: ${best_cost:.2f}")
        return best_solution, best_cost, cost_history

In [ ]:
def create_grasp_instance():
    """Create the example depot, customers, and vehicles."""
    depot = Depot(
        x=0,
        y=0,
        initial_inventory=500,
        holding_cost=0.1,
    )

    np.random.seed(42)
    random.seed(42)
    customers = []
    angles = np.linspace(0, 2 * np.pi, 8, endpoint=False)
    distances = [10, 15, 20, 25, 12, 18, 22, 28]

    for index, (angle, distance) in enumerate(zip(angles, distances)):
        customer = Customer(
            id=index + 1,
            x=distance * np.cos(angle),
            y=distance * np.sin(angle),
            demand_mean=np.random.uniform(5, 15),
            demand_std=np.random.uniform(1, 3),
            holding_cost=np.random.uniform(0.1, 0.3),
            initial_inventory=np.random.uniform(20, 40),
            min_inventory=np.random.uniform(5, 10),
            max_inventory=np.random.uniform(40, 60),
        )
        customers.append(customer)

    vehicles = [
        Vehicle(id=1, capacity=50, fixed_cost=50, variable_cost=1.0),
        Vehicle(id=2, capacity=50, fixed_cost=50, variable_cost=1.0),
    ]
    return depot, customers, vehicles


print("Creating the GRASP IRP instance...")
depot, customers, vehicles = create_grasp_instance()

print(f"Depot location: ({depot.x:.1f}, {depot.y:.1f})")
print(f"Initial depot inventory: {depot.initial_inventory:.1f} units")
print("\nCustomers:")
for customer in customers:
    print(
        f"  C{customer.id}: location=({customer.x:.1f}, {customer.y:.1f}), "
        f"mean demand={customer.demand_mean:.1f} units/period, "
        f"demand standard deviation={customer.demand_std:.1f}, "
        f"initial inventory={customer.initial_inventory:.1f} units"
    )

print("\nVehicles:")
for vehicle in vehicles:
    print(
        f"  Vehicle {vehicle.id}: capacity={vehicle.capacity:.1f} units, "
        f"fixed route cost=${vehicle.fixed_cost:.2f}, "
        f"variable cost=${vehicle.variable_cost:.2f}/distance unit"
    )

In [ ]:
grasp = GRASPIRP(
    depot=depot,
    customers=customers,
    vehicles=vehicles,
    num_periods=5,
    alpha=0.3,
    max_iterations=50,
)

print("\nSolving the IRP with GRASP...")
start_time = time.time()
best_solution, best_cost, cost_history = grasp.solve()
elapsed_time = time.time() - start_time

print(f"\nElapsed computation time: {elapsed_time:.2f} seconds")
print(f"Best observed total cost: ${best_cost:.2f}")

# Check that every constructed delivery respects storage, vehicle, and depot limits.
for period_routes in best_solution:
    period_delivery_total = 0.0
    for route in period_routes:
        vehicle = next(item for item in vehicles if item.id == route.vehicle_id)
        route_delivery_total = sum(delivery.quantity for delivery in route.deliveries)
        assert route_delivery_total <= vehicle.capacity + 1e-9
        assert len(route.customers) == len(route.deliveries)
        assert all(0.0 < delivery.quantity <= 10.0 for delivery in route.deliveries)
        period_delivery_total += route_delivery_total
    assert period_delivery_total <= depot.initial_inventory + 1e-9

In [ ]:
def analyze_grasp_solution(solution, cost_history):
    """Summarize the best observed solution and iteration history."""
    if not solution or not cost_history:
        raise ValueError("A solution and a nonempty cost history are required.")

    print("\n=== GRASP SOLUTION ANALYSIS ===")
    print(f"Best-so-far cost after the first iteration: ${cost_history[0]:.2f}")
    print(f"Best observed cost at termination: ${cost_history[-1]:.2f}")
    if cost_history[0] != 0.0:
        improvement = (cost_history[0] - cost_history[-1]) / cost_history[0] * 100
        print(f"Improvement in best-so-far cost: {improvement:.1f}%")

    total_routes = 0
    total_deliveries = 0
    total_distance = 0.0

    print("\nSolution structure:")
    for period, routes in enumerate(solution, start=1):
        print(f"\nPeriod {period}:")
        total_routes += len(routes)
        for route in routes:
            delivery_summary = [
                (delivery.customer_id, round(delivery.quantity, 2))
                for delivery in route.deliveries
            ]
            print(f"  Vehicle {route.vehicle_id}: route {route.customers}")
            print(
                f"    Distance: {route.total_distance:.2f} distance units; "
                f"transportation cost: ${route.total_cost:.2f}"
            )
            print(f"    Deliveries (customer, units): {delivery_summary}")
            total_distance += route.total_distance
            total_deliveries += len(route.deliveries)

    print("\nSummary:")
    print(f"Total routes: {total_routes}")
    print(f"Total customer deliveries: {total_deliveries}")
    print(f"Total distance: {total_distance:.2f} distance units")
    print(f"Average distance per route: {total_distance / max(1, total_routes):.2f}")
    print(
        "Average customer deliveries per route: "
        f"{total_deliveries / max(1, total_routes):.2f}"
    )


analyze_grasp_solution(best_solution, cost_history)

In [ ]:
def visualize_grasp_results(
    depot,
    customers,
    vehicles,
    solution,
    cost_history,
):
    """Plot the solution, best-so-far costs, and RCL parameter experiment."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle("GRASP IRP Solution Analysis", fontsize=16, fontweight="bold")

    axes[0, 0].plot(cost_history, marker="o", linewidth=2, color="blue")
    axes[0, 0].set_title("Best-So-Far Cost by GRASP Iteration")
    axes[0, 0].set_xlabel("Iteration")
    axes[0, 0].set_ylabel("Total cost ($)")
    axes[0, 0].grid(True, alpha=0.3)

    route_axis = axes[0, 1]
    route_axis.scatter(
        depot.x, depot.y, c="red", s=200, marker="s", label="Depot", zorder=5
    )
    for customer in customers:
        route_axis.scatter(
            customer.x, customer.y, c="blue", s=100, marker="o", zorder=4
        )
        route_axis.text(customer.x + 1, customer.y + 1, f"C{customer.id}", fontsize=9)

    route_colors = ["green", "orange", "purple", "brown"]
    if solution and solution[0]:
        for index, route in enumerate(solution[0]):
            color = route_colors[index % len(route_colors)]
            route_axis.plot([], [], color=color, linewidth=2, label=f"Vehicle {route.vehicle_id}")
            coordinates = [(depot.x, depot.y)]
            for customer_id in route.customers:
                customer = next(item for item in customers if item.id == customer_id)
                coordinates.append((customer.x, customer.y))
            coordinates.append((depot.x, depot.y))
            for start, end in zip(coordinates, coordinates[1:]):
                route_axis.plot(
                    [start[0], end[0]],
                    [start[1], end[1]],
                    color=color,
                    linewidth=2,
                    alpha=0.7,
                )

    route_axis.set_title("Routes in Period 1")
    route_axis.set_xlabel("X coordinate (distance units)")
    route_axis.set_ylabel("Y coordinate (distance units)")
    route_axis.legend()
    route_axis.grid(True, alpha=0.3)

    periods = list(range(1, len(solution) + 1))
    routes_per_period = [len(period_routes) for period_routes in solution]
    deliveries_per_period = [
        sum(len(route.deliveries) for route in period_routes)
        for period_routes in solution
    ]
    axes[1, 0].bar(periods, routes_per_period, alpha=0.7, color="skyblue")
    axes[1, 0].set_title("Routes per Period")
    axes[1, 0].set_xlabel("Period")
    axes[1, 0].set_ylabel("Number of routes")
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].bar(periods, deliveries_per_period, alpha=0.7, color="lightcoral")
    axes[1, 1].set_title("Customer Deliveries per Period")
    axes[1, 1].set_xlabel("Period")
    axes[1, 1].set_ylabel("Number of deliveries")
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    alpha_values = [0.1, 0.3, 0.5, 0.7, 0.9]
    alpha_costs = []
    alpha_times = []
    for alpha in alpha_values:
        grasp_test = GRASPIRP(
            depot=depot,
            customers=customers,
            vehicles=vehicles,
            num_periods=5,
            alpha=alpha,
            max_iterations=20,
        )
        start_time = time.time()
        _, cost, _ = grasp_test.solve()
        alpha_times.append(time.time() - start_time)
        alpha_costs.append(cost)

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    axes[0].plot(alpha_values, alpha_costs, marker="o", linewidth=2, color="blue")
    axes[0].set_title("Observed Cost by RCL Parameter")
    axes[0].set_xlabel("Alpha (α)")
    axes[0].set_ylabel("Best observed total cost ($)")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(alpha_values, alpha_times, marker="s", linewidth=2, color="red")
    axes[1].set_title("Computation Time by RCL Parameter")
    axes[1].set_xlabel("Alpha (α)")
    axes[1].set_ylabel("Computation time (seconds)")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


visualize_grasp_results(depot, customers, vehicles, best_solution, cost_history)

In [ ]:
def performance_analysis():
    """Measure observed costs and runtimes on generated instance sizes."""
    print("\n=== PERFORMANCE ANALYSIS ===")
    problem_sizes = [4, 6, 8, 10]
    costs = []
    times = []

    for size in problem_sizes:
        print(f"\nTesting an instance with {size} customers")
        angles = np.linspace(0, 2 * np.pi, size, endpoint=False)
        distances = np.random.uniform(10, 30, size)
        test_customers = []

        for index, (angle, distance) in enumerate(zip(angles, distances)):
            test_customers.append(
                Customer(
                    id=index + 1,
                    x=distance * np.cos(angle),
                    y=distance * np.sin(angle),
                    demand_mean=np.random.uniform(5, 15),
                    demand_std=np.random.uniform(1, 3),
                    holding_cost=np.random.uniform(0.1, 0.3),
                    initial_inventory=np.random.uniform(20, 40),
                    min_inventory=np.random.uniform(5, 10),
                    max_inventory=np.random.uniform(40, 60),
                )
            )

        grasp_test = GRASPIRP(
            depot=depot,
            customers=test_customers,
            vehicles=vehicles,
            num_periods=3,
            alpha=0.3,
            max_iterations=30,
        )
        start_time = time.time()
        _, cost, _ = grasp_test.solve()
        elapsed = time.time() - start_time
        costs.append(cost)
        times.append(elapsed)
        print(f"  Best observed cost: ${cost:.2f}")
        print(f"  Elapsed time: {elapsed:.2f} seconds")

    fig, (cost_axis, time_axis) = plt.subplots(1, 2, figsize=(15, 6))
    cost_axis.plot(problem_sizes, costs, marker="o", linewidth=2, color="blue")
    cost_axis.set_title("Observed Cost by Number of Customers")
    cost_axis.set_xlabel("Number of customers")
    cost_axis.set_ylabel("Best observed total cost ($)")
    cost_axis.grid(True, alpha=0.3)

    time_axis.plot(problem_sizes, times, marker="s", linewidth=2, color="red")
    time_axis.set_title("Computation Time by Number of Customers")
    time_axis.set_xlabel("Number of customers")
    time_axis.set_ylabel("Computation time (seconds)")
    time_axis.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    if costs[0] != 0.0 and times[0] != 0.0:
        print("\nObserved endpoint ratios:")
        print(
            f"Cost ratio, {problem_sizes[0]} to {problem_sizes[-1]} customers: "
            f"{costs[-1] / costs[0]:.2f}x"
        )
        print(
            f"Time ratio, {problem_sizes[0]} to {problem_sizes[-1]} customers: "
            f"{times[-1] / times[0]:.2f}x"
        )


performance_analysis()

## Interpreting the Experiments

### What the cost history shows

The plotted history records the best cost observed up to each GRASP iteration. It therefore cannot increase. A flat segment means that subsequent iterations did not find a lower-cost solution; it does not prove that the current solution is globally optimal. Because each iteration simulates new random demand, the best observed cost is scenario-dependent and should not be read as an estimate of average performance.

### Understanding the RCL experiment

The experiment varies `alpha` while keeping the instance and iteration budget fixed. The plotted costs and runtimes are observations from this run, not universal performance guarantees. A restricted candidate list emphasizes the highest-ranked candidates; a broader list allows more random choices. Any apparent cost difference may also reflect the stochastic demand realizations and random selections encountered during the experiment.

### Understanding the instance-size experiment

The performance experiment generates instances with different customer counts and records the heuristic's observed cost and elapsed time. These measurements depend on the generated data, machine, and run. They illustrate behavior on these instances; they do not establish a general complexity rate or a guaranteed runtime for larger problems.

### Strengths and limitations

GRASP combines greedy guidance with randomized construction and local improvement. Repeating the construction-and-search process provides opportunities to find different solutions, while retaining the best observed solution prevents the reported best-so-far cost from worsening. The method is useful when a practical solution is needed without an optimality certificate.

The heuristic's quality depends on its construction score, RCL parameter, iteration budget, and local-search neighborhood. The 2-opt operator changes customer order within a route; it does not reassign customers between vehicles or optimize delivery quantities beyond the construction rule. The example's simplified inventory and demand treatment also limits how directly its results can be applied to operational planning.